# 02 - Data Preprocessing

Clean, convert to ShareGPT format, merge, and split datasets.

**Output:** `data/splits/train.json`, `val.json`, `test.json`

In [ ]:
%%capture
!pip install unsloth datasets

In [ ]:
import os
os.chdir('/kaggle/working')

# Clone the project repo (or upload src/ as a Kaggle dataset)
if not os.path.exists('fingpt-qlora'):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir('fingpt-qlora')
print(f"Working directory: {os.getcwd()}")

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset
import matplotlib.pyplot as plt

## 1. Download Datasets

In [ ]:
from src.data.download import download_all

download_all()
print("\nDownloaded files:")
for f in Path('data/raw').glob('*.json'):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

## 2. Preprocess (Clean & Deduplicate)

In [ ]:
from src.data.preprocess import preprocess_all

preprocess_all()

## 3. Convert to ShareGPT Format

In [ ]:
from src.data.format_chat import convert_all

conversations = convert_all()
print(f"\nTotal conversations: {len(conversations)}")

In [ ]:
# Inspect 3 examples
import random
random.seed(42)
for ex in random.sample(conversations, 3):
    print(f"\nTask: {ex.get('task_type', 'unknown')}")
    for turn in ex['conversations']:
        print(f"  [{turn['from']}]: {turn['value'][:150]}...")
    print("---")

## 4. Merge Datasets with Stratified Weights

In [ ]:
from src.data.merge_datasets import merge_all

merged = merge_all()
print(f"\nMerged dataset size: {len(merged)}")

## 5. Train/Val/Test Split

In [ ]:
from src.data.splits import split_all

split_all()

# Verify
for split in ['train', 'val', 'test']:
    path = Path(f'data/splits/{split}.json')
    with open(path) as f:
        data = json.load(f)
    print(f"\n{split}: {len(data)} examples")
    
    # Task distribution
    from collections import Counter
    tasks = Counter(ex.get('task_type', 'unknown') for ex in data)
    for task, count in sorted(tasks.items()):
        print(f"  {task}: {count} ({100*count/len(data):.1f}%)")

## 6. Validation Checks

In [ ]:
# Structure validation
with open('data/splits/train.json') as f:
    train_data = json.load(f)

errors = 0
for i, ex in enumerate(train_data):
    if 'conversations' not in ex:
        print(f"  Error at {i}: missing 'conversations'")
        errors += 1
        continue
    conv = ex['conversations']
    if len(conv) < 2:
        print(f"  Error at {i}: too few turns ({len(conv)})")
        errors += 1
    if conv[0]['from'] != 'system':
        print(f"  Error at {i}: first turn should be system, got {conv[0]['from']}")
        errors += 1
    if conv[1]['from'] != 'human':
        print(f"  Error at {i}: second turn should be human, got {conv[1]['from']}")
        errors += 1
    if conv[-1]['from'] != 'gpt':
        print(f"  Error at {i}: last turn should be gpt, got {conv[-1]['from']}")
        errors += 1

print(f"\nValidation: {errors} errors found in {len(train_data)} examples")
if errors == 0:
    print("All checks passed!")

In [ ]:
# Final task distribution visualization
from collections import Counter

tasks = Counter(ex.get('task_type', 'unknown') for ex in train_data)
plt.figure(figsize=(8, 5))
plt.bar(tasks.keys(), tasks.values(), color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6'])
plt.title('Training Data - Task Distribution')
plt.ylabel('Count')
for i, (k, v) in enumerate(tasks.items()):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()